## 4.2 自动微分 - 梯度更新案例

#### 1. 案例背景：

我们设定一个非常简单的模型：
* ŷ = w * b 
* 给一条训练数据：x = 2, y = 4
* 显然 w = 2

损失函数用最常见的 MSE（平方误差）：
* Loss = (y - ŷ)^2
* 训练目标则是将Loss不断减小，使得 w 接近2

#### 2. 案例一：一次梯度更新 - 手动

##### 2.1 准备数据

In [15]:
import torch
x = torch.tensor(2.0)
y = torch.tensor(4.0)

##### 2.2 准备参数 初始w
* 加上 requires_grad = True 表示这个w参数是需要使用自动求导的参数，需要计算梯度
* 给定一个初始的 w

In [16]:
w = torch.tensor(1.0, requires_grad=True)

##### 2.3 forward：预测值与损失值

In [17]:
y_pred = w * x
loss = (y_pred - y) ** 2

print("Before update: ")
print("w: ", w.item(), "loss: ", loss.item())

Before update: 
w:  1.0 loss:  4.0


##### 2.4 backword: 计算loss对w的偏导数 d(loss)/d(w) **（梯度）**

In [18]:
loss.backward()
gradient = w.grad
print("Gradient: ", gradient.item())

Gradient:  -8.0


#### 2.5 更新参数w

注意：
* 非常关键：更新时不要让 autograd 继续追踪
* 因为 w -= lr*w.grad 也是一次运算。
* 如果不加 no_grad()：
    * PyTorch 会继续把这次更新也记录进计算图
    * 计算图会越来越大（内存爆炸）
    * 并且下一轮 backward 会变得混乱


⚠️注意：
* 更新 w 参数时，不能使用 w = w - lr * gradient
    * 这行代码创建了一个新的 tensor 并赋值给 w，新的 w 是一个普通 tensor，不再有 requires_grad=True
    * 需要在更新完之后补充上 w.requires_grad = True，或者：👇
* 解决方法：应该使用 原地操作（in-place） w -= ... 或 w.data -=，保留原始 tensor

In [19]:
# 学习率设置为0.01
lr = 0.01
with torch.no_grad():
    w -= lr * gradient # 原地修改w，w还是原来的tensor对象，保持requires_grad=True
    # 或者：
    # w = w - lr * gradient
    # w.requires_grad_(True)

#### 2.6 清空梯度，否则梯度会累加

In [20]:
w.grad.zero_()

tensor(0.)

##### 2.7 再算一次看看 loss 是否下降

In [21]:
y_pred_2 = w * x
loss_2 = (y_pred_2 - y) ** 2
print("After update: ")
print("w: ", w.item(), "loss: ", loss_2.item())

After update: 
w:  1.0800000429153442 loss:  3.3855996131896973


#### 3. 案例二：循环更新梯度（循环训练的雏形）

这就是以后写神经网络训练循环的原型：forward → loss → backward → update → zero_grad

##### 训练30次，查看w是否逼近2

In [22]:
# 1. 准备数据 x, y
x = torch.tensor(2.0)
y = torch.tensor(4.0)

# 2. 初始化参数 w
w = torch.tensor(1.0, requires_grad=True)

# 3 设置学习率
lr = 0.01

# 4. 创建迭代的训练循环
epochs = 30
for epoch in range(epochs):
    # 5. forward pass: 计算预测值和损失
    y_pre = w * x
    loss = (y_pre - y) ** 2
    print("Epoch {}: w = {}, loss = {}".format(epoch+1, w.item(), loss.item()))

    # 6. backward pass: 计算梯度
    loss.backward()
    gradient = w.grad
    print("Epoch {}: gradient = {}".format(epoch+1, gradient.item()))

    # 7. 更新参数
    with torch.no_grad():
        w -= lr * gradient

    # 8. 清零梯度
    w.grad.zero_()

Epoch 1: w = 1.0, loss = 4.0
Epoch 1: gradient = -8.0
Epoch 2: w = 1.0800000429153442, loss = 3.3855996131896973
Epoch 2: gradient = -7.359999656677246
Epoch 3: w = 1.1535999774932861, loss = 2.865571975708008
Epoch 3: gradient = -6.771200180053711
Epoch 4: w = 1.2213119268417358, loss = 2.4254205226898193
Epoch 4: gradient = -6.229504585266113
Epoch 5: w = 1.283607006072998, loss = 2.0528757572174072
Epoch 5: gradient = -5.731143951416016
Epoch 6: w = 1.3409184217453003, loss = 1.7375540733337402
Epoch 6: gradient = -5.272652626037598
Epoch 7: w = 1.39364492893219, loss = 1.4706659317016602
Epoch 7: gradient = -4.8508405685424805
Epoch 8: w = 1.4421533346176147, loss = 1.2447715997695923
Epoch 8: gradient = -4.462773323059082
Epoch 9: w = 1.486781120300293, loss = 1.0535744428634644
Epoch 9: gradient = -4.105751037597656
Epoch 10: w = 1.5278385877609253, loss = 0.8917455673217773
Epoch 10: gradient = -3.7772912979125977
Epoch 11: w = 1.565611481666565, loss = 0.7547735571861267
Epoch 

#### 4. 把它和深度学习训练闭环对齐（重要）

现在写的循环其实就是：
1. forward：算预测
2. compute loss：算损失
3. backward：算梯度
4. update：更新参数
5. zero_grad：清空梯度

以后你换成神经网络也只是：
1. y_hat = model(x)
2. loss = criterion(y_hat, y)
3. loss.backward()
4. optimizer.step()
5. optimizer.zero_grad()